# PISA 2022 & HSLS:09 Interactive Visualizations

This notebook generates 9 interactive Vega-Lite visualizations with linked driving/driven charts:
- **Viz 1-3: PISA-only charts** (international education context)
- **Viz 4-6: HSLS-only charts** (US longitudinal data)
- **Viz 7-9: Combined charts** (US vs international comparison)

In [ ]:
import pandas as pd
import altair as alt
from pathlib import Path
import json

DATA_DIR = Path("../ComprehensiveAnalysisOnHSLS/data")
OUTPUT_DIR = Path("../assets/json")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pisa_df = pd.read_csv(DATA_DIR / "pisa_subset.csv", low_memory=False)
hsls_df = pd.read_csv(DATA_DIR / "hsls_subset.csv", low_memory=False)

print(f"PISA: {len(pisa_df)} rows, {len(pisa_df.columns)} columns")
print(f"HSLS: {len(hsls_df)} rows, {len(hsls_df.columns)} columns")

In [ ]:
DARK_CONFIG = {
    "background": "#030712",
    "view": {"stroke": "transparent"},
    "axis": {
        "labelColor": "#E0E0E0",
        "titleColor": "#FFFFFF",
        "gridColor": "#333333",
        "domainColor": "#444444",
        "tickColor": "#444444"
    },
    "legend": {
        "labelColor": "#E0E0E0",
        "titleColor": "#FFFFFF"
    },
    "title": {"color": "#FFFFFF"}
}

OECD_COUNTRIES = ["AUS", "AUT", "BEL", "CAN", "CHE", "CHL", "COL", "CRI", "CZE", "DEU",
                  "DNK", "ESP", "EST", "FIN", "FRA", "GBR", "GRC", "HUN", "IRL", "ISL",
                  "ISR", "ITA", "JPN", "KOR", "LTU", "LUX", "LVA", "MEX", "NLD", "NOR",
                  "NZL", "POL", "PRT", "SVK", "SVN", "SWE", "TUR", "USA"]

def save_chart(chart, filename):
    spec = json.loads(chart.to_json())
    spec["config"] = DARK_CONFIG
    with open(OUTPUT_DIR / filename, "w") as f:
        json.dump(spec, f, indent=2)
    print(f"Saved: {filename}")

---
## PISA-ONLY VISUALIZATIONS (Viz 1-3)
---

### Viz 1: Gender Gap in Math Self-Efficacy by Country (Grouped Bar)

**Left (Driver):** Grouped bar chart showing gender scores by country  
**Right (Driven):** Bar chart showing gender scores for selected country

In [ ]:
pisa_gender = pisa_df[pisa_df["MATHEFF"].notna() & pisa_df["ST004D01T"].isin([1, 2])].copy()
pisa_gender["Gender"] = pisa_gender["ST004D01T"].map({1: "Female", 2: "Male"})

gender_efficacy = pisa_gender.groupby(["CNT", "Gender"]).agg({"MATHEFF": "mean"}).reset_index()
gender_efficacy.columns = ["Country", "Gender", "Math_Efficacy"]

gender_pivot = gender_efficacy.pivot(index="Country", columns="Gender", values="Math_Efficacy").reset_index()
gender_pivot["Gap"] = gender_pivot["Male"] - gender_pivot["Female"]
top_20 = gender_pivot.nlargest(20, "Gap")

gender_long = top_20.melt(id_vars=["Country", "Gap"], value_vars=["Female", "Male"], 
                          var_name="Gender", value_name="Math_Efficacy")

country_select = alt.selection_point(fields=["Country"], name="country_select")

left_chart = alt.Chart(gender_long).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("Math_Efficacy:Q", title="Math Self-Efficacy", scale=alt.Scale(domain=[-0.8, 0.8])),
    y=alt.Y("Country:N", sort=alt.EncodingSortField(field="Gap", order="descending"), title=None),
    color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]),
                    legend=alt.Legend(title="Gender", orient="top")),
    xOffset="Gender:N",
    opacity=alt.condition(country_select, alt.value(1), alt.value(0.3)),
    tooltip=["Country:N", "Gender:N", alt.Tooltip("Math_Efficacy:Q", format=".2f")]
).add_params(country_select).properties(
    name="view_1",
    title={"text": "Click Country for Gap Detail", "color": "#FFFFFF", "fontSize": 14},
    width=400, height=400
)

right_chart = alt.Chart(gender_long).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4).encode(
    x=alt.X("Math_Efficacy:Q", title="Math Self-Efficacy Score", scale=alt.Scale(domain=[-0.8, 0.8])),
    y=alt.Y("Gender:N", title=None),
    color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]),
                    legend=alt.Legend(title="Gender", orient="top")),
    tooltip=["Country:N", "Gender:N", alt.Tooltip("Math_Efficacy:Q", format=".2f")]
).transform_filter(country_select).properties(
    title={"text": "Gender Scores for Selected Country", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=400
)

viz1 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz1, "pisa_gender_efficacy_dumbbell.json")
viz1

### Viz 2: Math Anxiety vs Performance (Heatmap)

**Left (Driver):** Heatmap of anxiety level vs math level  
**Right (Driven):** Bar chart showing count by anxiety level for selected cell

In [ ]:
pisa_anxiety = pisa_df[pisa_df["ANXMAT"].notna() & pisa_df["PV1MATH"].notna()].copy()
pisa_anxiety["Anxiety_Level"] = pd.qcut(pisa_anxiety["ANXMAT"], 5, labels=["Very Low", "Low", "Medium", "High", "Very High"])
pisa_anxiety["Math_Level"] = pd.qcut(pisa_anxiety["PV1MATH"], 5, labels=["Level 1", "Level 2", "Level 3", "Level 4", "Level 5"])

heatmap_data = pisa_anxiety.groupby(["Anxiety_Level", "Math_Level"]).size().reset_index(name="Count")
total = heatmap_data["Count"].sum()
heatmap_data["Percentage"] = (heatmap_data["Count"] / total * 100).round(2)

cell_select = alt.selection_point(fields=["Anxiety_Level", "Math_Level"], name="cell_select")

math_level_colors = ["#d73027", "#fc8d59", "#fee090", "#91bfdb", "#4575b4"]

left_chart = alt.Chart(heatmap_data).mark_rect(cursor="pointer").encode(
    x=alt.X("Math_Level:O", title="Math Performance Level", sort=["Level 1", "Level 2", "Level 3", "Level 4", "Level 5"]),
    y=alt.Y("Anxiety_Level:O", title="Math Anxiety Level", sort=["Very High", "High", "Medium", "Low", "Very Low"]),
    color=alt.Color("Count:Q", scale=alt.Scale(scheme="oranges"), legend=alt.Legend(title="Student Count")),
    opacity=alt.condition(cell_select, alt.value(1), alt.value(0.6)),
    tooltip=["Anxiety_Level:N", "Math_Level:N", alt.Tooltip("Count:Q", format=",d"), alt.Tooltip("Percentage:Q", format=".1f", title="%")]
).add_params(cell_select).properties(
    name="view_1",
    title={"text": "Click Cell for Details", "color": "#FFFFFF", "fontSize": 14},
    width=400, height=400
)

right_chart = alt.Chart(heatmap_data).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4).encode(
    x=alt.X("Count:Q", title="Student Count"),
    y=alt.Y("Anxiety_Level:N", title=None, sort=["Very High", "High", "Medium", "Low", "Very Low"]),
    color=alt.Color("Math_Level:N", 
                    scale=alt.Scale(domain=["Level 1", "Level 2", "Level 3", "Level 4", "Level 5"], 
                                    range=math_level_colors),
                    legend=alt.Legend(title="Math Level", orient="top")),
    tooltip=["Anxiety_Level:N", "Math_Level:N", alt.Tooltip("Count:Q", format=",d"), alt.Tooltip("Percentage:Q", format=".1f")]
).transform_filter(cell_select).properties(
    title={"text": "Selected Cell Details", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=400
)

viz2 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz2, "pisa_anxiety_performance_heatmap.json")
viz2

### Viz 3: Immigration Status & Achievement by Region

**Left (Driver):** Bar chart of average math score by region  
**Right (Driven):** Bar chart of achievement by immigration status for selected region

In [ ]:
pisa_immig = pisa_df[pisa_df["IMMIG"].isin([1, 2, 3]) & pisa_df["PV1MATH"].notna()].copy()
pisa_immig["Immigration_Status"] = pisa_immig["IMMIG"].map({1: "Native", 2: "Second-gen", 3: "First-gen"})

def get_region(cnt):
    if cnt == "USA":
        return "United States"
    elif cnt in OECD_COUNTRIES:
        return "OECD (excl. US)"
    else:
        return "Non-OECD"

pisa_immig["Region"] = pisa_immig["CNT"].apply(get_region)

region_agg = pisa_immig.groupby("Region").agg({"PV1MATH": "mean", "CNT": "count"}).reset_index()
region_agg.columns = ["Region", "Math_Score", "Count"]

immig_agg = pisa_immig.groupby(["Region", "Immigration_Status"]).agg({"PV1MATH": "mean", "CNT": "count"}).reset_index()
immig_agg.columns = ["Region", "Immigration_Status", "Math_Score", "Count"]

region_select = alt.selection_point(fields=["Region"], name="region_select")

left_chart = alt.Chart(region_agg).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("Math_Score:Q", title="Average Math Score", scale=alt.Scale(domain=[350, 550])),
    y=alt.Y("Region:N", title=None, sort=["United States", "OECD (excl. US)", "Non-OECD"]),
    color=alt.Color("Region:N", 
                    scale=alt.Scale(domain=["United States", "OECD (excl. US)", "Non-OECD"], range=["#E69F00", "#56B4E9", "#009E73"]),
                    legend=alt.Legend(title="Region", orient="top")),
    opacity=alt.condition(region_select, alt.value(1), alt.value(0.3)),
    tooltip=["Region:N", alt.Tooltip("Math_Score:Q", format=".0f"), alt.Tooltip("Count:Q", format=",d")]
).add_params(region_select).properties(
    name="view_1",
    title={"text": "Click Region to See Immigration Detail", "color": "#FFFFFF", "fontSize": 14},
    width=400, height=400
)

right_chart = alt.Chart(immig_agg).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4).encode(
    x=alt.X("Math_Score:Q", title="Average Math Score", scale=alt.Scale(domain=[350, 550])),
    y=alt.Y("Immigration_Status:N", title=None, sort=["Native", "Second-gen", "First-gen"]),
    color=alt.Color("Immigration_Status:N", 
                    scale=alt.Scale(domain=["Native", "Second-gen", "First-gen"], range=["#0072B2", "#56B4E9", "#D55E00"]),
                    legend=alt.Legend(title="Immigration Status", orient="top")),
    tooltip=["Region:N", "Immigration_Status:N", alt.Tooltip("Math_Score:Q", format=".0f"), alt.Tooltip("Count:Q", format=",d")]
).transform_filter(region_select).properties(
    title={"text": "Achievement by Immigration Status for Selected Region", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=400
)

viz3 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz3, "combined_immigration.json")
viz3

---
## HSLS-ONLY VISUALIZATIONS (Viz 4-6)
---

### Viz 4: Parent Education & STEM Expectations by Gender

**Left (Driver):** Bar chart of STEM rate by parent education with gradient colors  
**Right (Driven):** Bar chart of STEM rate by gender for selected education level

In [ ]:
hsls_stem = hsls_df[hsls_df["X1PAR1EDU"].notna() & hsls_df["X1SEX"].notna() & hsls_df["X1STU30OCC_STEM1"].notna()].copy()
hsls_stem = hsls_stem[hsls_stem["X1PAR1EDU"] > 0]
hsls_stem = hsls_stem[hsls_stem["X1SEX"].isin([1, 2])]

edu_map = {1: "Less than HS", 2: "HS Diploma", 3: "Some College", 4: "Associate's", 5: "Bachelor's", 6: "Master's", 7: "Ph.D./Prof"}
hsls_stem["Parent_Education"] = hsls_stem["X1PAR1EDU"].map(edu_map)
hsls_stem["Gender"] = hsls_stem["X1SEX"].map({1: "Male", 2: "Female"})

edu_agg = hsls_stem.groupby("Parent_Education").agg({"X1STU30OCC_STEM1": "mean", "X1SEX": "count"}).reset_index()
edu_agg.columns = ["Parent_Education", "STEM_Rate", "Count"]
edu_agg["STEM_Rate"] = edu_agg["STEM_Rate"] * 100

gender_edu_agg = hsls_stem.groupby(["Parent_Education", "Gender"]).agg({"X1STU30OCC_STEM1": "mean", "X1SEX": "count"}).reset_index()
gender_edu_agg.columns = ["Parent_Education", "Gender", "STEM_Rate", "Count"]
gender_edu_agg["STEM_Rate"] = gender_edu_agg["STEM_Rate"] * 100

edu_order = ["Less than HS", "HS Diploma", "Some College", "Associate's", "Bachelor's", "Master's", "Ph.D./Prof"]
edu_colors = ["#e1bee7", "#ce93d8", "#ba68c8", "#ab47bc", "#9c27b0", "#7b1fa2", "#6a1b9a"]

edu_select = alt.selection_point(fields=["Parent_Education"], name="edu_select")

left_chart = alt.Chart(edu_agg).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("STEM_Rate:Q", title="Overall STEM Rate (%)", scale=alt.Scale(domain=[0, 50])),
    y=alt.Y("Parent_Education:N", title=None, sort=edu_order),
    color=alt.Color("Parent_Education:N", 
                    scale=alt.Scale(domain=edu_order, range=edu_colors),
                    legend=alt.Legend(title="Parent Education", orient="top", columns=4)),
    opacity=alt.condition(edu_select, alt.value(1), alt.value(0.3)),
    tooltip=["Parent_Education:N", alt.Tooltip("STEM_Rate:Q", format=".1f"), alt.Tooltip("Count:Q", format=",d")]
).add_params(edu_select).properties(
    name="view_1",
    title={"text": "Click Education Level for Gender Detail", "color": "#FFFFFF", "fontSize": 14},
    width=400, height=400
)

right_chart = alt.Chart(gender_edu_agg).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4).encode(
    x=alt.X("STEM_Rate:Q", title="STEM Career Expectation Rate (%)", scale=alt.Scale(domain=[0, 50])),
    y=alt.Y("Gender:N", title=None),
    color=alt.Color("Gender:N", 
                    scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]),
                    legend=alt.Legend(title="Gender", orient="top")),
    tooltip=["Parent_Education:N", "Gender:N", alt.Tooltip("STEM_Rate:Q", format=".1f"), alt.Tooltip("Count:Q", format=",d")]
).transform_filter(edu_select).properties(
    title={"text": "Gender Gap for Selected Education Level", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=400
)

viz4 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz4, "combined_gender_stem.json")
viz4

### Viz 5: Math Identity & Achievement by Race

**Left (Driver):** Bar chart of average math identity by race  
**Right (Driven):** Box plot of math identity distribution for selected race

In [ ]:
hsls_identity = hsls_df[hsls_df["X1MTHID"].notna() & hsls_df["X1TXMTSCOR"].notna() & hsls_df["X1RACE"].notna()].copy()
hsls_identity = hsls_identity[hsls_identity["X1RACE"] > 0]
hsls_identity = hsls_identity[(hsls_identity["X1MTHID"] >= -3) & (hsls_identity["X1MTHID"] <= 3)]
hsls_identity = hsls_identity[hsls_identity["X1TXMTSCOR"] >= 0]

race_map = {
    1: "Am. Indian/Alaska Native", 2: "Asian", 3: "Black/African American",
    4: "Hispanic", 5: "More than one race", 6: "Native Hawaiian/Pacific Islander", 7: "White"
}
hsls_identity["Race"] = hsls_identity["X1RACE"].map(race_map)

race_agg = hsls_identity.groupby("Race").agg({"X1MTHID": "mean", "X1TXMTSCOR": "count"}).reset_index()
race_agg.columns = ["Race", "Math_Identity", "Count"]

box_data = hsls_identity[["Race", "X1MTHID", "X1TXMTSCOR"]].copy()
box_data.columns = ["Race", "Math_Identity", "Math_Score"]

race_colors = ["#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7"]
race_order = list(race_map.values())

race_select = alt.selection_point(fields=["Race"], name="race_select")

left_chart = alt.Chart(race_agg).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("Math_Identity:Q", title="Average Math Identity Score", scale=alt.Scale(domain=[-0.5, 0.5])),
    y=alt.Y("Race:N", title=None, sort=race_order),
    color=alt.Color("Race:N", scale=alt.Scale(domain=race_order, range=race_colors), 
                    legend=alt.Legend(title="Race/Ethnicity", orient="top", columns=3)),
    opacity=alt.condition(race_select, alt.value(1), alt.value(0.3)),
    tooltip=["Race:N", alt.Tooltip("Math_Identity:Q", format=".2f"), alt.Tooltip("Count:Q", format=",d")]
).add_params(race_select).properties(
    name="view_1",
    title={"text": "Click Race for Identity Distribution", "color": "#FFFFFF", "fontSize": 14},
    width=400, height=400
)

right_chart = alt.Chart(box_data).mark_boxplot(extent="min-max", size=40).encode(
    x=alt.X("Math_Identity:Q", title="Math Identity Score Distribution", scale=alt.Scale(domain=[-3, 3])),
    y=alt.Y("Race:N", title=None, sort=race_order),
    color=alt.Color("Race:N", scale=alt.Scale(domain=race_order, range=race_colors),
                    legend=alt.Legend(title="Race/Ethnicity", orient="top", columns=3))
).transform_filter(race_select).properties(
    title={"text": "Identity Distribution for Selected Race", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=400
)

viz5 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz5, "hsls_math_identity_race.json")
viz5

### Viz 6: GPA Trajectories by SES Quintile

**Left (Driver):** Bar chart of 12th grade GPA by SES quintile  
**Right (Driven):** Line chart of GPA trajectory for selected SES quintile

In [ ]:
gpa_cols = ["X3TGPA9TH", "X3TGPA10TH", "X3TGPA11TH", "X3TGPA12TH"]
hsls_gpa = hsls_df[hsls_df["X1SESQ5"].notna()].copy()
hsls_gpa = hsls_gpa[hsls_gpa["X1SESQ5"] > 0]

for col in gpa_cols:
    hsls_gpa = hsls_gpa[hsls_gpa[col].notna() & (hsls_gpa[col] > 0)]

ses_labels = {1: "Lowest 20%", 2: "Second 20%", 3: "Middle 20%", 4: "Fourth 20%", 5: "Highest 20%"}
hsls_gpa["SES_Quintile"] = hsls_gpa["X1SESQ5"].map(ses_labels)

ses_12th = hsls_gpa.groupby("SES_Quintile").agg({"X3TGPA12TH": "mean", "X1SESQ5": "count"}).reset_index()
ses_12th.columns = ["SES_Quintile", "GPA", "Count"]

gpa_long = hsls_gpa.melt(id_vars=["SES_Quintile"], value_vars=gpa_cols, var_name="Grade", value_name="GPA")
grade_map = {"X3TGPA9TH": "9th Grade", "X3TGPA10TH": "10th Grade", "X3TGPA11TH": "11th Grade", "X3TGPA12TH": "12th Grade"}
gpa_long["Grade"] = gpa_long["Grade"].map(grade_map)
gpa_trajectory = gpa_long.groupby(["SES_Quintile", "Grade"]).agg({"GPA": "mean"}).reset_index()

ses_order = ["Lowest 20%", "Second 20%", "Middle 20%", "Fourth 20%", "Highest 20%"]
ses_colors = ["#1a237e", "#303f9f", "#3f51b5", "#7986cb", "#c5cae9"]

ses_select = alt.selection_point(fields=["SES_Quintile"], name="ses_select")

left_chart = alt.Chart(ses_12th).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("GPA:Q", title="Average 12th Grade GPA", scale=alt.Scale(domain=[2.0, 3.5])),
    y=alt.Y("SES_Quintile:N", title=None, sort=ses_order),
    color=alt.Color("SES_Quintile:N", scale=alt.Scale(domain=ses_order, range=ses_colors), 
                    legend=alt.Legend(title="SES Quintile", orient="top", columns=3)),
    opacity=alt.condition(ses_select, alt.value(1), alt.value(0.3)),
    tooltip=["SES_Quintile:N", alt.Tooltip("GPA:Q", format=".2f"), alt.Tooltip("Count:Q", format=",d")]
).add_params(ses_select).properties(
    name="view_1",
    title={"text": "Click SES Level for GPA Trajectory", "color": "#FFFFFF", "fontSize": 14},
    width=400, height=400
)

grade_order = ["9th Grade", "10th Grade", "11th Grade", "12th Grade"]

right_chart = alt.Chart(gpa_trajectory).mark_line(point={"size": 100}, strokeWidth=3).encode(
    x=alt.X("Grade:O", title="Grade Level", sort=grade_order),
    y=alt.Y("GPA:Q", title="Average GPA", scale=alt.Scale(domain=[2.0, 3.5])),
    color=alt.Color("SES_Quintile:N", scale=alt.Scale(domain=ses_order, range=ses_colors),
                    legend=alt.Legend(title="SES Quintile", orient="top", columns=3)),
    tooltip=["SES_Quintile:N", "Grade:O", alt.Tooltip("GPA:Q", format=".2f")]
).transform_filter(ses_select).properties(
    title={"text": "GPA Trajectory for Selected SES Level", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=400
)

viz6 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz6, "hsls_gpa_ses_trajectory.json")
viz6

---
## COMBINED VISUALIZATIONS (Viz 7-9)
---

### Viz 7: US vs International Math Self-Efficacy Comparison

**Left (Driver):** Bar chart of mean efficacy by data source  
**Right (Driven):** Grouped bar chart of gender comparison for selected source

In [ ]:
hsls_eff = hsls_df[hsls_df["X1MTHEFF"].notna() & hsls_df["X1SEX"].isin([1, 2])].copy()
hsls_eff["Source"] = "HSLS (US 2009)"
hsls_eff["Gender"] = hsls_eff["X1SEX"].map({1: "Male", 2: "Female"})
hsls_eff["Efficacy_Z"] = (hsls_eff["X1MTHEFF"] - hsls_eff["X1MTHEFF"].mean()) / hsls_eff["X1MTHEFF"].std()
hsls_subset = hsls_eff[["Source", "Gender", "Efficacy_Z"]].copy()

pisa_eff = pisa_df[pisa_df["MATHEFF"].notna() & pisa_df["ST004D01T"].isin([1, 2])].copy()
pisa_eff["Gender"] = pisa_eff["ST004D01T"].map({1: "Female", 2: "Male"})
pisa_eff["Efficacy_Z"] = (pisa_eff["MATHEFF"] - pisa_eff["MATHEFF"].mean()) / pisa_eff["MATHEFF"].std()

pisa_us = pisa_eff[pisa_eff["CNT"] == "USA"].copy()
pisa_us["Source"] = "PISA US (2022)"
pisa_us_subset = pisa_us[["Source", "Gender", "Efficacy_Z"]].copy()

pisa_intl = pisa_eff[pisa_eff["CNT"] != "USA"].copy()
pisa_intl["Source"] = "PISA Intl (2022)"
pisa_intl_subset = pisa_intl[["Source", "Gender", "Efficacy_Z"]].copy()

combined = pd.concat([hsls_subset, pisa_us_subset, pisa_intl_subset])

source_agg = combined.groupby("Source").agg({"Efficacy_Z": "mean", "Gender": "count"}).reset_index()
source_agg.columns = ["Source", "Mean_Efficacy", "Count"]

source_gender_agg = combined.groupby(["Source", "Gender"]).agg({"Efficacy_Z": "mean"}).reset_index()
source_gender_agg.columns = ["Source", "Gender", "Mean_Efficacy"]

source_order = ["HSLS (US 2009)", "PISA US (2022)", "PISA Intl (2022)"]
source_colors = ["#E69F00", "#56B4E9", "#009E73"]

source_select = alt.selection_point(fields=["Source"], name="source_select")

left_chart = alt.Chart(source_agg).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("Mean_Efficacy:Q", title="Mean Standardized Efficacy", scale=alt.Scale(domain=[-0.5, 0.5])),
    y=alt.Y("Source:N", title=None, sort=source_order),
    color=alt.Color("Source:N", scale=alt.Scale(domain=source_order, range=source_colors), 
                    legend=alt.Legend(title="Data Source", orient="top")),
    opacity=alt.condition(source_select, alt.value(1), alt.value(0.3)),
    tooltip=["Source:N", alt.Tooltip("Mean_Efficacy:Q", format=".2f"), alt.Tooltip("Count:Q", format=",d")]
).add_params(source_select).properties(
    name="view_1",
    title={"text": "Click Source for Gender Detail", "color": "#FFFFFF", "fontSize": 14},
    width=400, height=400
)

right_chart = alt.Chart(source_gender_agg).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4).encode(
    x=alt.X("Mean_Efficacy:Q", title="Mean Standardized Efficacy", scale=alt.Scale(domain=[-0.5, 0.5])),
    y=alt.Y("Gender:N", title=None),
    color=alt.Color("Gender:N", scale=alt.Scale(domain=["Female", "Male"], range=["#E91E63", "#1976D2"]),
                    legend=alt.Legend(title="Gender", orient="top")),
    xOffset="Source:N",
    tooltip=["Source:N", "Gender:N", alt.Tooltip("Mean_Efficacy:Q", format=".2f")]
).transform_filter(source_select).properties(
    title={"text": "Gender Gap for Selected Source", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=400
)

viz7 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz7, "combined_efficacy_comparison.json")
viz7

### Viz 8: SES-Achievement Gradient Comparison

**Left (Driver):** Bar chart of student count by data source  
**Right (Driven):** Line chart of SES gradient for selected source

In [ ]:
hsls_ses = hsls_df[hsls_df["X1SESQ5"].notna() & hsls_df["X1TXMTSCOR"].notna()].copy()
hsls_ses = hsls_ses[hsls_ses["X1SESQ5"] > 0]
hsls_ses["SES_Quintile"] = hsls_ses["X1SESQ5"].astype(int)
hsls_ses["Math_Z"] = (hsls_ses["X1TXMTSCOR"] - hsls_ses["X1TXMTSCOR"].mean()) / hsls_ses["X1TXMTSCOR"].std()
hsls_ses["Source"] = "HSLS (US 2009)"

pisa_ses = pisa_df[pisa_df["ESCS"].notna() & pisa_df["PV1MATH"].notna()].copy()
pisa_ses["SES_Quintile"] = pd.qcut(pisa_ses["ESCS"], 5, labels=[1, 2, 3, 4, 5]).astype(int)
pisa_ses["Math_Z"] = (pisa_ses["PV1MATH"] - pisa_ses["PV1MATH"].mean()) / pisa_ses["PV1MATH"].std()

pisa_us_ses = pisa_ses[pisa_ses["CNT"] == "USA"].copy()
pisa_us_ses["Source"] = "PISA US (2022)"

pisa_intl_ses = pisa_ses[pisa_ses["CNT"] != "USA"].copy()
pisa_intl_ses["Source"] = "PISA Intl (2022)"

combined = pd.concat([hsls_ses[["Source", "SES_Quintile", "Math_Z"]], 
                      pisa_us_ses[["Source", "SES_Quintile", "Math_Z"]], 
                      pisa_intl_ses[["Source", "SES_Quintile", "Math_Z"]]])

source_count = combined.groupby("Source").size().reset_index(name="Count")
ses_gradient = combined.groupby(["Source", "SES_Quintile"]).agg({"Math_Z": "mean"}).reset_index()

source_order = ["HSLS (US 2009)", "PISA US (2022)", "PISA Intl (2022)"]
source_colors = ["#E69F00", "#56B4E9", "#009E73"]

source_select = alt.selection_point(fields=["Source"], name="source_select")

left_chart = alt.Chart(source_count).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("Count:Q", title="Number of Students", scale=alt.Scale(domain=[0, 700000])),
    y=alt.Y("Source:N", title=None, sort=source_order),
    color=alt.Color("Source:N", scale=alt.Scale(domain=source_order, range=source_colors), 
                    legend=alt.Legend(title="Data Source", orient="top")),
    opacity=alt.condition(source_select, alt.value(1), alt.value(0.3)),
    tooltip=["Source:N", alt.Tooltip("Count:Q", format=",d")]
).add_params(source_select).properties(
    name="view_1",
    title={"text": "Click Source for SES Gradient", "color": "#FFFFFF", "fontSize": 14},
    width=400, height=400
)

right_chart = alt.Chart(ses_gradient).mark_line(point={"size": 100}, strokeWidth=3).encode(
    x=alt.X("SES_Quintile:O", title="SES Quintile (1=Lowest, 5=Highest)"),
    y=alt.Y("Math_Z:Q", title="Standardized Math Score", scale=alt.Scale(domain=[-1.0, 1.0])),
    color=alt.Color("Source:N", scale=alt.Scale(domain=source_order, range=source_colors),
                    legend=alt.Legend(title="Data Source", orient="top")),
    tooltip=["Source:N", "SES_Quintile:O", alt.Tooltip("Math_Z:Q", format=".2f")]
).transform_filter(source_select).properties(
    title={"text": "SES Gradient for Selected Source", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=400
)

viz8 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz8, "combined_ses_achievement.json")
viz8

### Viz 9: Parent Education Effect on Math Achievement

**Left (Driver):** Grouped bar chart of math achievement by parent education level  
**Right (Driven):** Bar chart comparing sources for selected education level

In [ ]:
hsls_edu_map = {1: "Less than HS", 2: "HS Diploma", 3: "Some College", 4: "Some College", 5: "Bachelor's", 6: "Graduate+", 7: "Graduate+"}
pisa_edu_map = {1: "Less than HS", 2: "Less than HS", 3: "HS Diploma", 4: "Some College", 5: "Some College", 6: "Bachelor's", 7: "Graduate+", 8: "Graduate+"}

hsls_edu = hsls_df[hsls_df["X1PAR1EDU"].notna() & hsls_df["X1TXMTSCOR"].notna()].copy()
hsls_edu = hsls_edu[hsls_edu["X1PAR1EDU"] > 0]
hsls_edu["Parent_Education"] = hsls_edu["X1PAR1EDU"].map(hsls_edu_map)
hsls_edu["Math_Z"] = (hsls_edu["X1TXMTSCOR"] - hsls_edu["X1TXMTSCOR"].mean()) / hsls_edu["X1TXMTSCOR"].std()
hsls_edu["Source"] = "HSLS (US 2009)"

pisa_edu = pisa_df[pisa_df["HISCED"].notna() & pisa_df["PV1MATH"].notna()].copy()
pisa_edu = pisa_edu[pisa_edu["HISCED"] > 0]
pisa_edu["Parent_Education"] = pisa_edu["HISCED"].map(pisa_edu_map)
pisa_edu["Math_Z"] = (pisa_edu["PV1MATH"] - pisa_edu["PV1MATH"].mean()) / pisa_edu["PV1MATH"].std()

pisa_us_edu = pisa_edu[pisa_edu["CNT"] == "USA"].copy()
pisa_us_edu["Source"] = "PISA US (2022)"

pisa_intl_edu = pisa_edu[pisa_edu["CNT"] != "USA"].copy()
pisa_intl_edu["Source"] = "PISA Intl (2022)"

combined = pd.concat([hsls_edu[["Source", "Parent_Education", "Math_Z"]], 
                      pisa_us_edu[["Source", "Parent_Education", "Math_Z"]], 
                      pisa_intl_edu[["Source", "Parent_Education", "Math_Z"]]])

edu_agg = combined.groupby(["Parent_Education", "Source"]).agg({"Math_Z": "mean"}).reset_index()
edu_agg.columns = ["Parent_Education", "Source", "Math_Score"]

edu_order = ["Less than HS", "HS Diploma", "Some College", "Bachelor's", "Graduate+"]
source_order = ["HSLS (US 2009)", "PISA US (2022)", "PISA Intl (2022)"]
source_colors = ["#E69F00", "#56B4E9", "#009E73"]

edu_select = alt.selection_point(fields=["Parent_Education"], name="edu_select")

left_chart = alt.Chart(edu_agg).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4, cursor="pointer").encode(
    x=alt.X("Math_Score:Q", title="Standardized Math Score", scale=alt.Scale(domain=[-1.0, 1.0])),
    y=alt.Y("Parent_Education:N", title=None, sort=edu_order),
    color=alt.Color("Source:N", scale=alt.Scale(domain=source_order, range=source_colors),
                    legend=alt.Legend(title="Data Source", orient="top")),
    xOffset="Source:N",
    opacity=alt.condition(edu_select, alt.value(1), alt.value(0.3)),
    tooltip=["Parent_Education:N", "Source:N", alt.Tooltip("Math_Score:Q", format=".2f")]
).add_params(edu_select).properties(
    name="view_1",
    title={"text": "Click Education Level for Source Comparison", "color": "#FFFFFF", "fontSize": 14},
    width=400, height=400
)

right_chart = alt.Chart(edu_agg).mark_bar(cornerRadiusTopRight=4, cornerRadiusBottomRight=4).encode(
    x=alt.X("Math_Score:Q", title="Standardized Math Score", scale=alt.Scale(domain=[-1.0, 1.0])),
    y=alt.Y("Source:N", title=None, sort=source_order),
    color=alt.Color("Source:N", scale=alt.Scale(domain=source_order, range=source_colors), 
                    legend=alt.Legend(title="Data Source", orient="top")),
    tooltip=["Parent_Education:N", "Source:N", alt.Tooltip("Math_Score:Q", format=".2f")]
).transform_filter(edu_select).properties(
    title={"text": "Source Comparison for Selected Education Level", "color": "#FFFFFF", "fontSize": 14},
    width=450, height=400
)

viz9 = alt.hconcat(left_chart, right_chart).resolve_scale(color="independent")
save_chart(viz9, "combined_parent_education.json")
viz9

---
## Summary

All 9 interactive visualizations have been generated:

**PISA-only (Viz 1-3):**
1. `pisa_gender_efficacy_dumbbell.json` - Gender Gap in Math Efficacy (Grouped Bar)
2. `pisa_anxiety_performance_heatmap.json` - Math Anxiety vs Performance
3. `combined_immigration.json` - Immigration & Achievement by Region

**HSLS-only (Viz 4-6):**
4. `combined_gender_stem.json` - Parent Ed & STEM by Gender (Gradient Colors)
5. `hsls_math_identity_race.json` - Math Identity & Achievement by Race (Box Plot)
6. `hsls_gpa_ses_trajectory.json` - GPA Trajectories by SES

**Combined (Viz 7-9):**
7. `combined_efficacy_comparison.json` - US vs Intl Math Efficacy
8. `combined_ses_achievement.json` - SES-Achievement Gradient
9. `combined_parent_education.json` - Parent Education Effect (xOffset Grouped)

In [ ]:
import os
json_files = [
    "pisa_gender_efficacy_dumbbell.json",
    "pisa_anxiety_performance_heatmap.json",
    "combined_immigration.json",
    "combined_gender_stem.json",
    "hsls_math_identity_race.json",
    "hsls_gpa_ses_trajectory.json",
    "combined_efficacy_comparison.json",
    "combined_ses_achievement.json",
    "combined_parent_education.json"
]

print("Generated JSON files:")
for f in json_files:
    path = OUTPUT_DIR / f
    if path.exists():
        size = os.path.getsize(path)
        print(f"  {f}: {size/1024:.1f} KB")
    else:
        print(f"  {f}: NOT FOUND")